# The Model That Doesn't Know
An image classifier that recognizes familiar categories and actively rejects unfamiliar or corrupted inputs as `UNKNOWN`.

- **Dataset:** CIFAR-10 (6 KNOWN classes for training, 4 Near-OOD UNKNOWN classes excluded) + **Far-OOD** (SVHN street digits & synthetic uniform noise).
- **Model:** ImageNet-1K pretrained `ResNet18`, fine-tuned on the 6 known classes (plus frozen-backbone ablation).
- **Uncertainty Methods (3 Families):**
  1. **Logit-Space:** Maximum Softmax Probability (`MSP`), Negative `Entropy`, `Energy`, `MaxLogit`
  2. **Penultimate Feature-Space:** `KNN-512D` (L2-normalized 512-D `avgpool` distance)
  3. **Hybrid Detector Fusion:** Z-score normalized `Energy + KNN-512D`
- **Extras:** Bootstrap 95% AUROC confidence intervals, Temperature Scaling calibration (`ECE`), 5-level Gaussian noise severity sweep, Risk-Coverage curves, Consensus failure analysis, and an interactive 3-state Gradio demo (`Known` / `Unknown Object` / `Degraded Image`).

In [11]:
import os, io, json, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.datasets as dset
from torch.utils.data import Dataset, DataLoader, Subset, TensorDataset
from PIL import Image, ImageFilter, ImageEnhance
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.neighbors import NearestNeighbors

## 0. Save everything to Google Drive
Checkpoints, results, and figures are saved in your Google Drive (`unknown_project`) so you never lose training progress if Colab disconnects.

In [12]:
OUT_DIR = '/content/unknown_project'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/unknown_project'
except Exception as e:
    print('Drive not mounted, saving locally only:', e)
os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = OUT_DIR + '/figures'
os.makedirs(FIG_DIR, exist_ok=True)
print('Saving to:', OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving to: /content/drive/MyDrive/unknown_project


## 1. Settings

In [13]:
SEED = 42
IMG_SIZE = 128
BATCH = 128
EPOCHS = 4
LR = 3e-4

ALL = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
KNOWN = ['airplane', 'automobile', 'ship', 'truck', 'cat', 'dog']
UNKNOWN = [c for c in ALL if c not in KNOWN]          # bird, deer, frog, horse
known_ids = [ALL.index(c) for c in KNOWN]
unknown_ids = [ALL.index(c) for c in UNKNOWN]
label_map = {orig: new for new, orig in enumerate(known_ids)}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| known:', KNOWN, '| unknown:', UNKNOWN)

device: cpu | known: ['airplane', 'automobile', 'ship', 'truck', 'cat', 'dog'] | unknown: ['bird', 'deer', 'frog', 'horse']


## 2. Dataset and Preprocessing
- **Train split (known classes only):** 27,000 images
- **Validation split (known classes only):** 3,000 images (used strictly to pick the 95% TPR thresholds and Temperature $T$)
- **Test Known:** 6,000 unseen images
- **Test Near-OOD (Unknown):** 4,000 images from the 4 excluded CIFAR classes
- **Test Far-OOD:** 4,000 SVHN digit images + 2,000 Uniform Noise sanity-check images

In [ ]:
base_train = torchvision.datasets.CIFAR10('./data', train=True, download=True)
base_test = torchvision.datasets.CIFAR10('./data', train=False, download=True)
train_targets = np.array(base_train.targets)
test_targets = np.array(base_test.targets)

rng = np.random.RandomState(SEED)
known_train_all = np.where(np.isin(train_targets, known_ids))[0]
rng.shuffle(known_train_all)
n_val = 3000
val_idx = known_train_all[:n_val]
train_idx = known_train_all[n_val:]
test_known_idx = np.where(np.isin(test_targets, known_ids))[0]
test_unk_idx = np.where(np.isin(test_targets, unknown_ids))[0]
print('train', len(train_idx), '| val', len(val_idx), '| test known', len(test_known_idx), '| test unknown', len(test_unk_idx))

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                      T.Resize(IMG_SIZE), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize(IMG_SIZE), T.ToTensor(), T.Normalize(MEAN, STD)])

class CifarSubset(Dataset):
    def __init__(self, base, indices, transform, pre=None):
        self.base, self.indices, self.transform, self.pre = base, indices, transform, pre
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, t = self.base[int(self.indices[i])]
        if self.pre is not None:
            img = self.pre(img)
        return self.transform(img), label_map.get(t, -1), t

def make_loader(base, idx, tf, shuffle=False, pre=None):
    return DataLoader(CifarSubset(base, idx, tf, pre), batch_size=BATCH, shuffle=shuffle, num_workers=2)

train_loader = make_loader(base_train, train_idx, train_tf, shuffle=True)
val_loader = make_loader(base_train, val_idx, eval_tf)
test_known_loader = make_loader(base_test, test_known_idx, eval_tf)
test_unk_loader = make_loader(base_test, test_unk_idx, eval_tf)

  1%|▏         | 2.29M/170M [01:12<1:29:48, 31.2kB/s]

## 3. Train (Fine-Tune ResNet18)
Automatically resumes from `checkpoint.pt` in Google Drive if already trained.

In [ ]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(KNOWN))
model = model.to(device)

@torch.no_grad()
def extract_all(net, loader):
    """Extracts 512-D penultimate features (avgpool), logits, mapped labels, and original CIFAR labels."""
    net.eval()
    feats, logits, ys, origs = [], [], [], []
    hook = net.avgpool.register_forward_hook(lambda m, i, o: feats.append(o.flatten(1).cpu()))
    for batch in loader:
        x = batch[0].to(device)
        logits.append(net(x).float().cpu())
        if len(batch) == 3:
            ys.append(batch[1])
            origs.append(batch[2])
    hook.remove()
    f_arr = torch.cat(feats).numpy()
    l_arr = torch.cat(logits).numpy()
    y_arr = torch.cat(ys).numpy() if ys else None
    o_arr = torch.cat(origs).numpy() if origs else None
    return f_arr, l_arr, y_arr, o_arr

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader))

CKPT = OUT_DIR + '/checkpoint.pt'
history, start_epoch = [], 0
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
    history, start_epoch = ck['history'], ck['epoch']
    print(f'Found saved checkpoint: {start_epoch} of {EPOCHS} epochs already done')

for epoch in range(start_epoch, EPOCHS):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y, _ in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss.backward(); opt.step(); sched.step()
        loss_sum += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); total += len(y)
    _, lv, yv, _ = extract_all(model, val_loader)
    val_acc = float((lv.argmax(1) == yv).mean())
    history.append({'epoch': epoch + 1, 'train_loss': loss_sum / total, 'train_acc': correct / total, 'val_acc': val_acc})
    print(history[-1])
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'history': history, 'epoch': epoch + 1}, CKPT)

torch.save(model.state_dict(), OUT_DIR + '/resnet18_known6.pt')

## 4. Evaluate on Unseen Images from Known Classes & Extract 512-D Features

In [ ]:
train_bank_idx = train_idx[:5000]
train_bank_loader = make_loader(base_train, train_bank_idx, eval_tf, shuffle=False)

feat_train, _, y_train_bank, _ = extract_all(model, train_bank_loader)
feat_val, logits_val, y_val, _ = extract_all(model, val_loader)
feat_known, logits_known, y_known, orig_known = extract_all(model, test_known_loader)
feat_unk, logits_unk, _, orig_unk = extract_all(model, test_unk_loader)

pred_known = logits_known.argmax(1)
known_acc = float((pred_known == y_known).mean())
print(f'Closed-Set Test Accuracy on 6 KNOWN classes: {known_acc:.4f}\n')
print(classification_report(y_known, pred_known, target_names=KNOWN, digits=3))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_known, pred_known), display_labels=KNOWN).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.tight_layout(); plt.savefig(FIG_DIR + '/confusion_matrix.png', dpi=150); plt.show()

## 5. Unified OOD Detection: Logit-Space vs. Feature-Space (`KNN-512D`), Fusion & Frozen-Backbone Ablation
- **Threshold Rule:** Picked strictly on the **validation set** (5th percentile = 95% target TPR).
- **Note on Threshold Columns:** `Near-OOD Rej % (Val Thr)` uses the validation threshold deployed in the system; `Near FPR@95 (Test Thr)` uses the test-known 5th percentile for standard literature comparison.

In [ ]:
def softmax_np(z, T=1.0):
    z = z / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def score_msp(logits, T=1.0): return softmax_np(logits, T).max(1)
def score_negentropy(logits, T=1.0):
    p = softmax_np(logits, T)
    return (p * np.log(p + 1e-12)).sum(1)
def score_energy(logits, T=1.0):
    z = logits / T
    m = z.max(1)
    return T * (m + np.log(np.exp(z - m[:, None]).sum(1)))
def score_maxlogit(logits, T=1.0): return logits.max(1)

SCORE_FNS = {'MSP': score_msp, 'Entropy': score_negentropy, 'Energy': score_energy, 'MaxLogit': score_maxlogit}

l2_norm = lambda f: f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-12)
knn_detector = NearestNeighbors(n_neighbors=10, metric='euclidean', n_jobs=-1).fit(l2_norm(feat_train))
score_knn = lambda f: -knn_detector.kneighbors(l2_norm(f))[0][:, -1]

# Far-OOD 1: SVHN (download=True ensures fresh Run All never fails)
svhn_full = dset.SVHN(root='./data', split='test', download=True, transform=eval_tf)
svhn_loader = DataLoader(Subset(svhn_full, range(4000)), batch_size=BATCH, shuffle=False, num_workers=2)
feat_svhn, logits_svhn, _, _ = extract_all(model, svhn_loader)

# Far-OOD 2 (Sanity Check): Synthetic Uniform Noise
rng_noise = torch.Generator().manual_seed(SEED)
noise_t = (torch.rand((2000, 3, IMG_SIZE, IMG_SIZE), generator=rng_noise) - torch.tensor(MEAN)[:, None, None]) / torch.tensor(STD)[:, None, None]
feat_noise, logits_noise, _, _ = extract_all(model, DataLoader(TensorDataset(noise_t), batch_size=BATCH))

def compute_all_scores(feats, logits):
    res = {m_name: score_fn(logits) for m_name, score_fn in SCORE_FNS.items()}
    res['KNN-512D'] = score_knn(feats)
    return res

sc_val   = compute_all_scores(feat_val, logits_val)
sc_known = compute_all_scores(feat_known, logits_known)
sc_near  = compute_all_scores(feat_unk, logits_unk)
sc_svhn  = compute_all_scores(feat_svhn, logits_svhn)
sc_noise = compute_all_scores(feat_noise, logits_noise)

# Detector Fusion: Z-score normalize Energy + KNN-512D on validation set
val_en_m, val_en_s = sc_val['Energy'].mean(), sc_val['Energy'].std() + 1e-12
val_kn_m, val_kn_s = sc_val['KNN-512D'].mean(), sc_val['KNN-512D'].std() + 1e-12
for d_sc in [sc_val, sc_known, sc_near, sc_svhn, sc_noise]:
    d_sc['Fusion (Energy+KNN)'] = 0.5 * ((d_sc['Energy'] - val_en_m) / val_en_s + (d_sc['KNN-512D'] - val_kn_m) / val_kn_s)

thresholds = {m_name: float(np.percentile(sc_val[m_name], 5)) for m_name in sc_val.keys()}

def bootstrap_auroc(sk, su, n_boot=200, seed=42):
    rng_b = np.random.RandomState(seed)
    y = np.r_[np.ones(len(sk)), np.zeros(len(su))]
    s = np.r_[sk, su]
    base_auc = roc_auc_score(y, s)
    aucs = [roc_auc_score(y[idx], s[idx]) for idx in (rng_b.choice(len(y), len(y), replace=True) for _ in range(n_boot))]
    return float(base_auc), float(np.std(aucs))

summary_rows = []
for m_name in sc_known.keys():
    thr_m = thresholds[m_name]
    sk, sn, ss, sno = sc_known[m_name], sc_near[m_name], sc_svhn[m_name], sc_noise[m_name]
    near_auc, near_std = bootstrap_auroc(sk, sn)
    svhn_auc, _ = bootstrap_auroc(sk, ss, n_boot=50)
    test_thr95 = np.percentile(sk, 5)
    summary_rows.append({
        'Method': m_name,
        'Family': 'Hybrid' if 'Fusion' in m_name else ('Feature-Space' if m_name == 'KNN-512D' else 'Logit-Space'),
        'Known Accepted %': round((sk >= thr_m).mean() * 100, 2),
        'Near-OOD Rej % (Val Thr)': round((sn < thr_m).mean() * 100, 2),
        'Near FPR@95 (Test Thr)': round((sn >= test_thr95).mean() * 100, 2),
        'Near AUROC (±Std)': f"{near_auc:.4f} ± {near_std:.4f}",
        'Near AUROC': round(near_auc, 4),
        'SVHN Rej %': round((ss < thr_m).mean() * 100, 2),
        'SVHN AUROC': round(svhn_auc, 4),
        'Noise Rej % (Sanity)': round((sno < thr_m).mean() * 100, 2),
    })

ood_table = pd.DataFrame(summary_rows).set_index('Method')
print("=== UNIFIED OOD BENCHMARK (Validation 95% TPR Threshold) ===")
print(ood_table.drop(columns=['Near AUROC']).to_string())

BEST = 'KNN-512D'
print(f"\nPrimary Operating Detector Selected: {BEST}")

# --- ABLATION: Frozen ImageNet ResNet18 Backbone (Before Fine-Tuning) vs Fine-Tuned ---
frozen_net = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1).to(device)
frozen_net.fc = nn.Identity()
f_tr_frz, _, _, _ = extract_all(frozen_net, train_bank_loader)
f_val_frz, _, _, _ = extract_all(frozen_net, val_loader)
f_kn_frz, _, _, _ = extract_all(frozen_net, test_known_loader)
f_un_frz, _, _, _ = extract_all(frozen_net, test_unk_loader)
del frozen_net

knn_frz = NearestNeighbors(n_neighbors=10, metric='euclidean', n_jobs=-1).fit(l2_norm(f_tr_frz))
sc_frz = lambda f: -knn_frz.kneighbors(l2_norm(f))[0][:, -1]
thr_frz = float(np.percentile(sc_frz(f_val_frz), 5))
frz_auc, frz_std = bootstrap_auroc(sc_frz(f_kn_frz), sc_frz(f_un_frz))
frz_rej = float((sc_frz(f_un_frz) < thr_frz).mean() * 100)
print(f"\n[ABLATION] Frozen ImageNet Backbone KNN-512D -> Near-OOD AUROC: {frz_auc:.4f} ± {frz_std:.4f} | Near-OOD Rej %: {frz_rej:.2f}%")
print(f"[ABLATION] Fine-Tuned ResNet18 KNN-512D      -> Near-OOD AUROC: {ood_table.loc['KNN-512D', 'Near AUROC (±Std)']} | Near-OOD Rej %: {ood_table.loc['KNN-512D', 'Near-OOD Rej % (Val Thr)']}%")

## 6. Confidence Distributions & ROC Curves: Known vs. Near-OOD vs. Far-OOD

In [ ]:
core_methods = ['MSP', 'Entropy', 'Energy', 'MaxLogit', 'KNN-512D']
fig, axes = plt.subplots(1, 5, figsize=(21, 3.6))
for ax, m_name in zip(axes, core_methods):
    sk, su = sc_known[m_name], sc_near[m_name]
    bins = np.linspace(min(sk.min(), su.min()), max(sk.max(), su.max()), 50)
    ax.hist(sk, bins=bins, alpha=0.6, label='Known (Test)', density=True)
    ax.hist(su, bins=bins, alpha=0.6, label='Near-OOD', density=True)
    ax.axvline(thresholds[m_name], color='k', linestyle='--', label='Val Threshold')
    ax.set_title(f"{m_name} (AUROC {ood_table.loc[m_name, 'Near AUROC']:.3f})")
    ax.set_xlabel('Score (Higher = Familiar)')
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR + '/confidence_histograms.png', dpi=150); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for m_name in core_methods + ['Fusion (Energy+KNN)']:
    y_roc = np.r_[np.ones(len(logits_known)), np.zeros(len(logits_unk))]
    fpr, tpr, _ = roc_curve(y_roc, np.r_[sc_known[m_name], sc_near[m_name]])
    axes[0].plot(fpr, tpr, label=f"{m_name} ({ood_table.loc[m_name, 'Near AUROC']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k:'); axes[0].set_xlabel('FPR (Unknown Accepted)'); axes[0].set_ylabel('TPR (Known Accepted)')
axes[0].legend(fontsize=8); axes[0].set_title('ROC Curves: Known vs Near-OOD')

for m_name, col in [('Entropy', 'tab:blue'), ('Energy', 'tab:orange'), ('KNN-512D', 'tab:green')]:
    sk, sn, ss = sc_known[m_name], sc_near[m_name], sc_svhn[m_name]
    if m_name == 'KNN-512D':
        bins = np.linspace(min(sk.min(), ss.min()), max(sk.max(), ss.max()), 50)
        axes[1].hist(sk, bins=bins, alpha=0.5, label='Known (CIFAR-10)', density=True)
        axes[1].hist(sn, bins=bins, alpha=0.5, label='Near-OOD (Animals)', density=True)
        axes[1].hist(ss, bins=bins, alpha=0.5, label='Far-OOD (SVHN)', density=True)
        axes[1].axvline(thresholds[m_name], color='k', linestyle='--', label='95% TPR Threshold')
        axes[1].set_title(f"KNN-512D Distributions (SVHN Rej = {ood_table.loc['KNN-512D', 'SVHN Rej %']}%)")
        axes[1].set_xlabel('KNN-512D Score'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR + '/roc_curves.png', dpi=150); plt.savefig(FIG_DIR + '/far_ood_comparison.png', dpi=150); plt.show()

## 7. Before vs. After Uncertainty Handling

In [ ]:
n_known, n_unk = len(logits_known), len(logits_unk)
comp_rows = [{
    'Setup': 'No uncertainty handling',
    'Known Accepted %': 100.0,
    'Accuracy on Accepted Known %': round(known_acc * 100, 2),
    'Near-OOD Rejected %': 0.0,
    'SVHN Rejected %': 0.0,
    'Overall System Accuracy %': round((pred_known == y_known).sum() / (n_known + n_unk) * 100, 2),
}]
for m_name in sc_known.keys():
    thr_m = thresholds[m_name]
    acc_mask = sc_known[m_name] >= thr_m
    unk_rej = sc_near[m_name] < thr_m
    svhn_rej = sc_svhn[m_name] < thr_m
    good = ((pred_known == y_known) & acc_mask).sum() + unk_rej.sum()
    comp_rows.append({
        'Setup': f'With {m_name}',
        'Known Accepted %': round(acc_mask.mean() * 100, 2),
        'Accuracy on Accepted Known %': round((pred_known[acc_mask] == y_known[acc_mask]).mean() * 100, 2),
        'Near-OOD Rejected %': round(unk_rej.mean() * 100, 2),
        'SVHN Rejected %': round(svhn_rej.mean() * 100, 2),
        'Overall System Accuracy %': round(good / (n_known + n_unk) * 100, 2),
    })
compare_table = pd.DataFrame(comp_rows).set_index('Setup')
print(compare_table.to_string())

best_unk_scores = sc_near[BEST]
best_thr = thresholds[BEST]
per_class = pd.DataFrame({'class': [ALL[i] for i in orig_unk], 'rejected': best_unk_scores < best_thr}).groupby('class')['rejected'].mean().mul(100).round(2)
print(f'\nRejection rate per unknown class ({BEST}):\n', per_class)

probs_unk = softmax_np(logits_unk)
pick = np.random.RandomState(1).choice(n_unk, 8, replace=False)
titles = [f"true: {ALL[orig_unk[p]]}\nbefore: {KNOWN[probs_unk[p].argmax()]} ({probs_unk[p].max():.0%})\nafter: {'UNKNOWN' if best_unk_scores[p] < best_thr else KNOWN[probs_unk[p].argmax()]}" for p in pick]

def show_grid(base, idx, titles, fname, ncols=4):
    n = len(idx); nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.8))
    axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis('off')
    for ax, i, t in zip(axes, idx, titles):
        ax.imshow(base.data[i]); ax.set_title(t, fontsize=8)
    plt.tight_layout(); plt.savefig(f'{FIG_DIR}/{fname}', dpi=150); plt.show()

show_grid(base_test, test_unk_idx[pick], titles, 'before_after_unknown.png')

## 8. Confidence Calibration (15-Bin ECE & Temperature Scaling)

In [ ]:
def ece_score(probs, labels, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    correct = (pred == labels).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def reliability_diagram(probs, labels, ax, title, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    correct = (pred == labels).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    accs, confs = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        accs.append(correct[m].mean() if m.any() else 0); confs.append((lo + hi) / 2)
    ax.bar(confs, accs, width=1 / n_bins, edgecolor='k', alpha=0.7, label='accuracy')
    ax.plot([0, 1], [0, 1], 'r--', label='perfect')
    ax.set_xlabel('confidence'); ax.set_ylabel('accuracy'); ax.set_title(title); ax.legend()

def fit_temperature(val_logits, val_labels):
    lg, lb = torch.tensor(val_logits), torch.tensor(val_labels)
    log_t = torch.zeros(1, requires_grad=True)
    optim = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        optim.zero_grad()
        loss = F.cross_entropy(lg / log_t.exp(), lb)
        loss.backward()
        return loss
    optim.step(closure)
    return float(log_t.exp().item())

TEMP = fit_temperature(logits_val, y_val)
p_before, p_after = softmax_np(logits_known), softmax_np(logits_known, TEMP)
ece_before, ece_after = ece_score(p_before, y_known, n_bins=15), ece_score(p_after, y_known, n_bins=15)
print(f'Temperature = {TEMP:.4f} | 15-Bin ECE before: {ece_before:.4f} | after: {ece_after:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
reliability_diagram(p_before, y_known, axes[0], f'Before (ECE {ece_before:.3f})', n_bins=15)
reliability_diagram(p_after, y_known, axes[1], f'After Temp Scaling (ECE {ece_after:.3f})', n_bins=15)
plt.tight_layout(); plt.savefig(FIG_DIR + '/reliability.png', dpi=150); plt.show()

## 9. Robustness to Image Corruptions, Noise Severity Sweep & Selective Risk-Coverage

In [ ]:
def c_noise(img):
    a = np.asarray(img).astype(np.float32) + np.random.normal(0, 25, np.asarray(img).shape)
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
def c_blur(img): return img.filter(ImageFilter.GaussianBlur(1.5))
def c_dark(img): return ImageEnhance.Brightness(img).enhance(0.3)
def c_jpeg(img):
    buf = io.BytesIO(); img.save(buf, 'JPEG', quality=10); buf.seek(0)
    return Image.open(buf).convert('RGB')

CORRUPTIONS = {'clean': None, 'gaussian noise': c_noise, 'blur': c_blur, 'dark (low light)': c_dark, 'jpeg quality 10': c_jpeg}
np.random.seed(SEED)
corr_rows = []
for c_name, fn_c in CORRUPTIONS.items():
    fc, lgc, yy, _ = extract_all(model, make_loader(base_test, test_known_idx, eval_tf, pre=fn_c))
    corr_rows.append({
        'corruption': c_name,
        'accuracy %': round((lgc.argmax(1) == yy).mean() * 100, 2),
        '15-bin ECE': round(ece_score(softmax_np(lgc), yy, n_bins=15), 4),
        'mean MSP conf': round(float(score_msp(lgc).mean()), 3),
        'rejected % (Entropy)': round((score_negentropy(lgc) < thresholds['Entropy']).mean() * 100, 2),
        'rejected % (Energy)': round((score_energy(lgc) < thresholds['Energy']).mean() * 100, 2),
        'rejected % (KNN-512D)': round((score_knn(fc) < thresholds['KNN-512D']).mean() * 100, 2),
    })
corr_table = pd.DataFrame(corr_rows).set_index('corruption')
print("=== CORRUPTION BENCHMARK ===")
print(corr_table.to_string())

sigmas = [0, 10, 20, 35, 50]
sweep_rows = []
for sig in sigmas:
    fn_s = None if sig == 0 else (lambda img, s=sig: Image.fromarray(
        np.clip(np.asarray(img).astype(np.float32) + np.random.normal(0, s, np.asarray(img).shape), 0, 255).astype(np.uint8)))
    np.random.seed(SEED)
    fc, lgc, _, _ = extract_all(model, make_loader(base_test, test_known_idx, eval_tf, pre=fn_s))
    sweep_rows.append({
        'Noise Sigma': sig,
        'Accuracy %': round((lgc.argmax(1) == y_known).mean() * 100, 2),
        'ECE (15-bin)': round(ece_score(softmax_np(lgc), y_known, n_bins=15), 4),
        'Energy Rej %': round((score_energy(lgc) < thresholds['Energy']).mean() * 100, 2),
        'KNN-512D Rej %': round((score_knn(fc) < thresholds['KNN-512D']).mean() * 100, 2),
    })
sweep_df = pd.DataFrame(sweep_rows).set_index('Noise Sigma')
print("\n=== GAUSSIAN NOISE SEVERITY SWEEP ===")
print(sweep_df.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(sigmas, sweep_df['Accuracy %'], 'o-', label='Classifier Accuracy %', color='tab:blue')
axes[0].plot(sigmas, sweep_df['Energy Rej %'], 's--', label='Energy Rejected %', color='tab:orange')
axes[0].plot(sigmas, sweep_df['KNN-512D Rej %'], '^--', label='KNN-512D Rejected %', color='tab:green')
axes[0].plot(sigmas, sweep_df['ECE (15-bin)'] * 100, 'd:', label='ECE x100 (Miscalibration)', color='tab:red')
axes[0].set_xlabel('Gaussian Noise Severity (Sigma)'); axes[0].set_ylabel('Percentage (%)')
axes[0].set_title('Corruption Sweep: Accuracy Drops, Calibration Breaks'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

coverages = np.linspace(0.1, 1.0, 50)
for m_name, color in [('MSP', 'tab:gray'), ('Entropy', 'tab:blue'), ('Energy', 'tab:orange'), ('KNN-512D', 'tab:green')]:
    sc_m = sc_known[m_name]
    order = np.argsort(-sc_m)
    correct_sorted = (pred_known == y_known)[order]
    axes[1].plot(coverages * 100, [correct_sorted[:int(c * len(sc_m))].mean() * 100 for c in coverages], label=m_name, color=color)
axes[1].set_xlabel('Coverage (% of Known Images Accepted)'); axes[1].set_ylabel('Accuracy on Accepted Known (%)')
axes[1].set_title('Selective Classification Risk-Coverage Curve'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR + '/severity_and_risk_coverage.png', dpi=150); plt.show()

## 10. Qualitative Error Analysis & Consensus Blind Spots

In [ ]:
# (a) Known images misclassified with high confidence
conf_known = softmax_np(logits_known).max(1)
wrong = np.where(pred_known != y_known)[0]
top_wrong = wrong[np.argsort(-conf_known[wrong])][:8]
titles = [f"true: {KNOWN[y_known[i]]}\npred: {KNOWN[pred_known[i]]} ({conf_known[i]:.0%})" for i in top_wrong]
show_grid(base_test, test_known_idx[top_wrong], titles, 'err_confident_wrong.png')

# (b) Detector failures on KNN-512D
accepted_unk = np.where(best_unk_scores >= best_thr)[0]
top_acc = accepted_unk[np.argsort(-best_unk_scores[accepted_unk])][:8]
pred_unk = logits_unk.argmax(1)
titles = [f"true: {ALL[orig_unk[i]]}\npred: {KNOWN[pred_unk[i]]}\nKNN {best_unk_scores[i]:.2f}" for i in top_acc]
show_grid(base_test, test_unk_idx[top_acc], titles, 'err_unknown_accepted.png')

# (c) Consensus Failures: Unknown images accepted by ALL 5 core detectors simultaneously
consensus_mask = np.ones(n_unk, dtype=bool)
for m_name in core_methods:
    consensus_mask &= (sc_near[m_name] >= thresholds[m_name])
print(f"Consensus Blind Spots (Accepted by ALL 5 detectors): {consensus_mask.sum()} / {n_unk} ({consensus_mask.mean()*100:.2f}%)")
df_cons = pd.DataFrame({'True Unknown': [ALL[i] for i in orig_unk[consensus_mask]], 'Predicted Known': [KNOWN[i] for i in pred_unk[consensus_mask]]})
print(pd.crosstab(df_cons['True Unknown'], df_cons['Predicted Known']))

# (d) False rejects on Known test set
best_known_scores = sc_known[BEST]
rej_known = np.where(best_known_scores < best_thr)[0]
print(f"\nKnown images rejected by {BEST}: {len(rej_known)} ({len(rej_known)/n_known:.1%}) | Base classifier error rate on rejected subset: {(pred_known[rej_known] != y_known[rej_known]).mean():.1%}")
worst_rej = rej_known[np.argsort(best_known_scores[rej_known])][:8]
titles = [f"true: {KNOWN[y_known[i]]}\npred: {KNOWN[pred_known[i]]}\nKNN {best_known_scores[i]:.2f}" for i in worst_rej]
show_grid(base_test, test_known_idx[worst_rej], titles, 'err_false_rejects.png')

## 11. Save Unified Results JSON

In [ ]:
results = {
    'known_test_accuracy': known_acc,
    'history': history,
    'best_method': BEST,
    'ood_table': ood_table.reset_index().to_dict('records'),
    'comparison_table': compare_table.reset_index().to_dict('records'),
    'thresholds': thresholds,
    'temperature': TEMP,
    'ece_before': ece_before,
    'ece_after': ece_after,
    'frozen_backbone_ablation': {'auroc': frz_auc, 'auroc_std': frz_std, 'near_ood_rej_pct': frz_rej},
    'consensus_failures_count': int(consensus_mask.sum()),
    'consensus_failures_pct': round(float(consensus_mask.mean() * 100), 2),
    'corruption_table': corr_table.reset_index().to_dict('records'),
    'noise_severity_sweep': sweep_df.reset_index().to_dict('records'),
    'per_unknown_class_rejection_pct': per_class.to_dict(),
}
with open(OUT_DIR + '/results.json', 'w') as f_out:
    json.dump(results, f_out, indent=2, default=float)
print(json.dumps(results, indent=2, default=float)[:2000])

## 12. Interactive Gradio Demo (`KNN-512D` + 3 Nearest Training Images + High-Frequency Noise Gate)
Separates inputs into 3 states: **Known Category**, **Unknown Object (`KNN-512D`)**, or **Degraded/Noisy Input**.

In [ ]:
!pip -q install gradio

In [ ]:
import gradio as gr

def estimate_noise_score(pil_img):
    arr = np.asarray(pil_img.convert('L').resize((128, 128)), dtype=np.float32)
    smooth = np.asarray(pil_img.convert('L').resize((128, 128)).filter(ImageFilter.BoxBlur(1)), dtype=np.float32)
    return float(np.std(arr - smooth))

@torch.no_grad()
def predict_demo(img, knn_thr_slider, noise_gate_slider):
    if img is None:
        return {}, 'Upload an image first.', None
    noise_level = estimate_noise_score(img)
    x = eval_tf(img.convert('RGB').resize((32, 32))).unsqueeze(0).to(device)
    feat_hook = []
    h = model.avgpool.register_forward_hook(lambda m, i, o: feat_hook.append(o.flatten(1).cpu().numpy()))
    lg = model(x).float().cpu().numpy()
    h.remove()
    probs = softmax_np(lg, TEMP)[0]
    f_norm = l2_norm(feat_hook[0])
    dists, nbr_indices = knn_detector.kneighbors(f_norm)
    knn_s = float(-dists[0, -1])

    fig_nbr, axes = plt.subplots(1, 3, figsize=(6.5, 2.3))
    for k_i, ax in enumerate(axes):
        orig_train_i = int(train_bank_idx[nbr_indices[0, k_i]])
        ax.imshow(base_train.data[orig_train_i])
        ax.set_title(f"#{k_i+1}: {KNOWN[y_train_bank[nbr_indices[0, k_i]]]}\ndist={dists[0, k_i]:.2f}", fontsize=9)
        ax.axis('off')
    plt.tight_layout()

    if noise_level > noise_gate_slider and knn_s < knn_thr_slider:
        verdict = f"DEGRADED / NOISY IMAGE REJECTED (Noise={noise_level:.1f} > {noise_gate_slider:.1f}, KNN={knn_s:.2f} < {knn_thr_slider:.2f})"
    elif knn_s < knn_thr_slider:
        verdict = f"UNKNOWN OBJECT REJECTED (KNN-512D score {knn_s:.2f} < threshold {knn_thr_slider:.2f})"
    else:
        verdict = f"ACCEPTED AS: {KNOWN[int(probs.argmax())].upper()} (KNN-512D score {knn_s:.2f} >= {knn_thr_slider:.2f})"
    return {KNOWN[i]: float(probs[i]) for i in range(len(KNOWN))}, verdict, fig_nbr

demo = gr.Interface(
    fn=predict_demo,
    inputs=[
        gr.Image(type='pil', label='Upload Image'),
        gr.Slider(minimum=-1.2, maximum=-0.2, value=round(thresholds['KNN-512D'], 3), step=0.01, label='KNN-512D Threshold (95% Val TPR Default)'),
        gr.Slider(minimum=5.0, maximum=35.0, value=16.0, step=0.5, label='High-Frequency Noise Gate Threshold')
    ],
    outputs=[
        gr.Label(num_top_classes=6, label='Calibrated Class Probabilities (Temp-Scaled)'),
        gr.Textbox(label='3-State System Verdict (Known / Unknown Object / Degraded Image)'),
        gr.Plot(label='3 Nearest Known Training Neighbors in 512-D Feature Space')
    ],
    title="The Model That Doesn't Know (KNN-512D + Noise Gate)",
    description='Known classes: ' + ', '.join(KNOWN) + '. Uses 512-D penultimate feature distance + high-frequency noise gating.'
)
demo.launch(share=True)